# ISIH Prototype — Sea-Ice Bias-Correction Model

**This trains the Sept 8 ISIH prototype, not the December production model.**
See `isih/README.md` in the repo for the difference.

What this does: learns to correct the GLORYS12 ocean-reanalysis sea-ice field
toward what satellites actually observed (NOAA/NSIDC CDR).

    corrected = clip(background + model_correction, 0, 1)

The model predicts the *correction*, never the field, so an untrained model
reproduces the background exactly and can't be much worse than it.

### Before running

1. **Enable internet**: Notebook settings → Internet → On
2. **Enable GPU**: Notebook settings → Accelerator → GPU T4 x2
3. **Add CMEMS credentials** as Kaggle Secrets (Add-ons → Secrets):
   - `CMEMS_USERNAME`
   - `CMEMS_PASSWORD`

   These are the same Copernicus Marine credentials used in the repo's GitHub
   Actions. NSIDC needs no credentials at all.


## 1. Environment

In [ ]:
!pip install -q copernicusmarine pyproj xarray netCDF4

import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Enable it in Notebook settings, or training will be ~20x slower.')

## 2. Project modules

In [ ]:
# Project modules, embedded from the repo so this notebook runs standalone.
# Generated by isih/kaggle/make_notebook.py -- do not edit here, edit the
# source files and regenerate, otherwise the two silently diverge.
from pathlib import Path
Path('isih_src').mkdir(exist_ok=True)

# ---- unet.py -----------------------------------------------------
Path('isih_src/unet.py').write_text('''"""Residual U-Net that bias-corrects the CMEMS sea-ice concentration forecast.

Predicts the residual (delta-SIC), never the field itself, so a zero output
reproduces the operational forecast exactly and the model's floor is CMEMS's
skill rather than zero. See docs/ML_ARCHITECTURE.md §1.1.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SICCorrectionUNet(nn.Module):
    """3-level U-Net. Measured parameter counts at depth=3, in_channels=19:

        base_filters=16 ->   485,041
        base_filters=24 -> 1,088,137
        base_filters=32 -> 1,931,617
        base_filters=40 -> 3,015,481   <- default, matches the ~3M the
                                          compute budget in §1.3 was costed on

    ML_ARCHITECTURE.md §1.1 pairs "16 base filters" with "~2-4M parameters";
    those two are inconsistent (16 filters gives 485K), so the default here
    follows the parameter count the compute plan actually depends on. Capacity
    should still be tuned empirically in Phase 2 — smaller may well be enough,
    and 485K would train ~6x faster.

    Input:  (B, in_channels, H, W) — the 19 channels of ML_ARCHITECTURE.md §1.2
    Output: (B, 1, H, W) — delta-SIC in SIC-fraction units, unbounded

    The output is deliberately unbounded: clipping to a valid concentration
    happens after adding the base forecast, in `apply_correction`, since it is
    the *sum* that must land in [0, 1], not the correction on its own.
    """

    def __init__(self, in_channels: int = 19, base_filters: int = 40, depth: int = 3):
        super().__init__()
        self.depth = depth

        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        ch = in_channels
        for level in range(depth):
            out_ch = base_filters * (2 ** level)
            self.encoders.append(ConvBlock(ch, out_ch))
            self.pools.append(nn.MaxPool2d(2))
            ch = out_ch

        self.bottleneck = ConvBlock(ch, base_filters * (2 ** depth))
        ch = base_filters * (2 ** depth)

        self.upsamples = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for level in reversed(range(depth)):
            skip_ch = base_filters * (2 ** level)
            self.upsamples.append(nn.ConvTranspose2d(ch, skip_ch, kernel_size=2, stride=2))
            self.decoders.append(ConvBlock(skip_ch * 2, skip_ch))
            ch = skip_ch

        self.head = nn.Conv2d(ch, 1, kernel_size=1)
        # Start as a no-op: an untrained model reproduces CMEMS exactly rather
        # than injecting random noise into an already-skilful forecast.
        nn.init.zeros_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_hw = x.shape[-2:]
        # Polar-stereographic grids (~332x316) are not divisible by 2^depth;
        # pad up so skip connections line up, then crop back at the end.
        x = self._pad_to_multiple(x, 2 ** self.depth)

        skips = []
        for encoder, pool in zip(self.encoders, self.pools):
            x = encoder(x)
            skips.append(x)
            x = pool(x)

        x = self.bottleneck(x)

        for upsample, decoder, skip in zip(self.upsamples, self.decoders, reversed(skips)):
            x = upsample(x)
            x = torch.cat([x, skip], dim=1)
            x = decoder(x)

        x = self.head(x)
        return x[..., : input_hw[0], : input_hw[1]]

    @staticmethod
    def _pad_to_multiple(x: torch.Tensor, multiple: int) -> torch.Tensor:
        h, w = x.shape[-2:]
        pad_h = (multiple - h % multiple) % multiple
        pad_w = (multiple - w % multiple) % multiple
        if pad_h or pad_w:
            x = F.pad(x, (0, pad_w, 0, pad_h), mode="replicate")
        return x


def apply_correction(base_forecast: torch.Tensor, delta: torch.Tensor) -> torch.Tensor:
    """corrected = clip(CMEMS forecast + predicted residual, 0, 1)."""
    return torch.clamp(base_forecast + delta, 0.0, 1.0)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)
''')

# ---- grid.py -----------------------------------------------------
Path('isih_src/grid.py').write_text('''"""NSIDC polar-stereographic grid geometry, and regridding GLORYS12 onto it.

Design decision (ML_ARCHITECTURE.md §4.2): the **truth field is never
resampled**. GLORYS12 is regridded onto the NSIDC grid, not the other way
round — resampling the field you score against would create ice-edge artifacts
indistinguishable from real model error, and IIEE is measured on that field.

GLORYS12 (~0.083°, ~9 km) is finer than NSIDC (25 km), so the correct
operation is **area-averaging** many GLORYS12 cells into each NSIDC cell, not
bilinear sampling. Bilinear would throw away most source cells and fail to
preserve ice area. Here that is done with a precomputed cell-index map plus a
bincount, so the expensive geometry runs once rather than per day.

CRS verified from a real NSIDC file (2026-08-31):
    EPSG:3412, Hughes 1980 ellipsoid, +proj=stere +lat_0=-90 +lat_ts=-70
Verified correct by transforming the grid origin, which returns lat = -90.0.
"""

from __future__ import annotations

import numpy as np
import xarray as xr
from pyproj import Transformer

NSIDC_CRS = "EPSG:3412"
LATLON_CRS = "EPSG:4326"


def nsidc_lonlat(ds: xr.Dataset) -> tuple[np.ndarray, np.ndarray]:
    """Return (lon, lat) arrays for every cell of an NSIDC grid, shape (y, x)."""
    transformer = Transformer.from_crs(NSIDC_CRS, LATLON_CRS, always_xy=True)
    x_grid, y_grid = np.meshgrid(ds.x.values, ds.y.values)
    lon, lat = transformer.transform(x_grid, y_grid)
    return lon, lat


class GlorysToNsidcRegridder:
    """Area-average GLORYS12 lat/lon fields onto the NSIDC polar grid.

    The mapping from source cells to target cells depends only on the two
    grids, so it is computed once at construction and reused for every day.
    """

    def __init__(
        self,
        nsidc_ds: xr.Dataset,
        source_lat: np.ndarray,
        source_lon: np.ndarray,
    ):
        self.ny = nsidc_ds.sizes["y"]
        self.nx = nsidc_ds.sizes["x"]
        self.n_cells = self.ny * self.nx

        # NSIDC grid is a regular 25 km grid in projected space, so a source
        # point's target cell is a direct arithmetic index — no search needed.
        x = nsidc_ds.x.values
        y = nsidc_ds.y.values
        self._x0 = float(x[0])
        self._y0 = float(y[0])
        self._dx = float(x[1] - x[0])
        self._dy = float(y[1] - y[0])

        src_lon_grid, src_lat_grid = np.meshgrid(source_lon, source_lat)
        transformer = Transformer.from_crs(LATLON_CRS, NSIDC_CRS, always_xy=True)
        src_x, src_y = transformer.transform(src_lon_grid, src_lat_grid)

        col = np.rint((src_x - self._x0) / self._dx).astype(np.int64)
        row = np.rint((src_y - self._y0) / self._dy).astype(np.int64)

        inside = (row >= 0) & (row < self.ny) & (col >= 0) & (col < self.nx)
        self._src_valid = inside.ravel()
        self._flat_index = (row * self.nx + col).ravel()[self._src_valid]

        # Cells with no source point at all must stay NaN rather than silently
        # reading as zero ice — an empty cell is unknown, not ice-free.
        self._counts_geometry = np.bincount(self._flat_index, minlength=self.n_cells)

    def __call__(self, source_values: np.ndarray) -> np.ndarray:
        """Regrid one (lat, lon) field to (y, x) on the NSIDC grid."""
        flat = source_values.ravel()[self._src_valid]
        finite = np.isfinite(flat)

        idx = self._flat_index[finite]
        vals = flat[finite]

        sums = np.bincount(idx, weights=vals, minlength=self.n_cells)
        counts = np.bincount(idx, minlength=self.n_cells)

        out = np.full(self.n_cells, np.nan, dtype=np.float32)
        has_data = counts > 0
        out[has_data] = (sums[has_data] / counts[has_data]).astype(np.float32)
        return out.reshape(self.ny, self.nx)

    def coverage(self) -> float:
        """Fraction of NSIDC cells any source point maps into. Sanity check:
        if this is low, the source subset does not cover the target grid."""
        return float((self._counts_geometry > 0).mean())
''')

import sys; sys.path.insert(0, 'isih_src')
print('modules written:', sorted(p.name for p in Path('isih_src').glob('*.py')))

## 3. Credentials

Read from Kaggle Secrets — never typed into a cell, so they are not saved into the notebook or its outputs.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ['COPERNICUSMARINE_SERVICE_USERNAME'] = secrets.get_secret('CMEMS_USERNAME')
os.environ['COPERNICUSMARINE_SERVICE_PASSWORD'] = secrets.get_secret('CMEMS_PASSWORD')
print('CMEMS credentials loaded from Kaggle Secrets')

## 4. Download the truth field (NSIDC)

No authentication needed. ~350 KB/day; one year is ~130 MB.

In [ ]:
import urllib.request, concurrent.futures, re
from pathlib import Path

YEARS = [2019, 2020]   # prototype scope; widen for the production model
BASE = 'https://noaadata.apps.nsidc.org/NOAA/G02202_V6/south/daily'
nsidc_dir = Path('data/nsidc'); nsidc_dir.mkdir(parents=True, exist_ok=True)

def year_files(year):
    with urllib.request.urlopen(f'{BASE}/{year}/', timeout=60) as r:
        html = r.read().decode('utf-8', 'replace')
    return sorted(set(re.findall(r'href="(sic_pss25_[^"]+\.nc)"', html)))

def fetch(year, name):
    dest = nsidc_dir / name
    if dest.exists() and dest.stat().st_size > 1000:
        return True
    urllib.request.urlretrieve(f'{BASE}/{year}/{name}', dest)
    return True

jobs = [(y, n) for y in YEARS for n in year_files(y)]
print(f'{len(jobs)} files to fetch')
with concurrent.futures.ThreadPoolExecutor(max_workers=8) as pool:
    list(pool.map(lambda a: fetch(*a), jobs))
print('downloaded:', len(list(nsidc_dir.glob('*.nc'))), 'files')

## 5. Download the background field (GLORYS12) and build training pairs

GLORYS12 is regridded onto the NSIDC grid by area-averaging — the truth field is never resampled, since resampling what you score against creates ice-edge artifacts indistinguishable from real model error.

In [ ]:
import numpy as np, xarray as xr, copernicusmarine
from grid import GlorysToNsidcRegridder

GLORYS = 'cmems_mod_glo_phy_my_0.083deg_P1D-m'
CORRIDOR = dict(min_lon=-5.0, max_lon=85.0, min_lat=-80.0, max_lat=-48.0)
gl_dir = Path('data/glorys'); gl_dir.mkdir(parents=True, exist_ok=True)

for year in YEARS:
    target = gl_dir / f'glorys_{year}.nc'
    if target.exists():
        print(year, 'already downloaded'); continue
    copernicusmarine.subset(
        dataset_id=GLORYS, variables=['siconc'],
        start_datetime=f'{year}-01-01T00:00:00', end_datetime=f'{year}-12-31T00:00:00',
        minimum_longitude=CORRIDOR['min_lon'], maximum_longitude=CORRIDOR['max_lon'],
        minimum_latitude=CORRIDOR['min_lat'], maximum_latitude=CORRIDOR['max_lat'],
        output_directory=str(gl_dir), output_filename=target.name, overwrite=True)
    print(year, 'done:', target.stat().st_size/1e6, 'MB')

In [ ]:
def load_nsidc_year(year):
    files = sorted(nsidc_dir.glob(f'sic_pss25_{year}*.nc'))
    truth, dates = [], []
    template = xr.open_dataset(files[0])
    for p in files:
        with xr.open_dataset(p) as ds:
            v = ds['cdr_seaice_conc'].isel(time=0).values.astype(np.float32)
            truth.append(np.where((v >= 0) & (v <= 1), v, np.nan))
            dates.append(int(p.name.split('_')[2]))
    return np.stack(truth), np.asarray(dates, np.int32), template

backgrounds, truths, masks, all_dates = [], [], [], []
for year in YEARS:
    truth, dates, template = load_nsidc_year(year)
    gl = xr.open_dataset(gl_dir / f'glorys_{year}.nc')
    rg = GlorysToNsidcRegridder(template, gl['latitude'].values, gl['longitude'].values)
    print(f'{year}: corridor covers {rg.coverage()*100:.1f}% of the NSIDC grid')
    gl_dates = np.array([int(s.replace('-','')) for s in
                         gl['time'].dt.strftime('%Y-%m-%d').values], np.int32)
    common, ni, gi = np.intersect1d(dates, gl_dates, return_indices=True)
    bg = np.stack([rg(gl['siconc'].isel(time=int(i)).values.astype(np.float32)) for i in gi])
    tr = truth[ni]
    backgrounds.append(bg); truths.append(tr)
    masks.append(np.isfinite(bg) & np.isfinite(tr)); all_dates.append(common)
    print(f'  {len(common)} paired days')

background = np.nan_to_num(np.concatenate(backgrounds), nan=0.0).astype(np.float32)
truth = np.nan_to_num(np.concatenate(truths), nan=0.0).astype(np.float32)
valid = np.concatenate(masks)
dates = np.concatenate(all_dates)
print('dataset:', background.shape, '| valid cells:', f'{valid.mean()*100:.1f}%')

## 6. Sanity check the pairing

Before training anything, confirm the two fields actually describe the same ocean. If background and truth were misaligned, correlation would be near zero and every later number would be meaningless.

In [ ]:
v = valid
corr = np.corrcoef(background[v], truth[v])[0, 1]
bias = float((background[v] - truth[v]).mean())
rmse = float(np.sqrt(((background[v] - truth[v])**2).mean()))
print(f'correlation background vs truth : {corr:.3f}')
print(f'mean bias (background - truth)   : {bias:+.4f}')
print(f'background RMSE vs truth         : {rmse:.4f}   <-- the bar to beat')
assert corr > 0.5, 'FIELDS ARE MISALIGNED — do not train on this'
print('\nAlignment OK. The trained model must beat the background RMSE above.')

## 7. Train

Temporal split — the last 20% of days is held out. A random split would leak adjacent days (ice fields are highly autocorrelated day to day) and inflate the score into meaninglessness.

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from unet import SICCorrectionUNet, apply_correction, count_parameters

order = np.argsort(dates)
background, truth, valid = background[order], truth[order], valid[order]
split = int(len(background) * 0.8)
print(f'train: {split} days | held-out: {len(background)-split} days')

def make_loader(sl, shuffle):
    x = torch.from_numpy(background[sl]).unsqueeze(1)
    y = torch.from_numpy(truth[sl]).unsqueeze(1)
    m = torch.from_numpy(valid[sl]).unsqueeze(1)
    return DataLoader(TensorDataset(x, y, m), batch_size=8, shuffle=shuffle)

train_loader = make_loader(slice(0, split), True)
val_loader   = make_loader(slice(split, None), False)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
# Prototype: 1 input channel (the background field) and a small network.
# The production model uses 19 channels -- see ML_ARCHITECTURE.md 1.2.
model = SICCorrectionUNet(in_channels=1, base_filters=16, depth=3).to(device)
print('parameters:', f'{count_parameters(model):,}')

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
scaler = torch.cuda.amp.GradScaler(enabled=(device=='cuda'))

def masked_mse(pred, target, mask):
    d = ((pred - target) ** 2) * mask
    return d.sum() / mask.sum().clamp(min=1)

EPOCHS = 30
best = float('inf')
for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb, mb in train_loader:
        xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device=='cuda')):
            corrected = apply_correction(xb, model(xb))
            loss = masked_mse(corrected, yb, mb)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()

    model.eval()
    se = n = base_se = 0.0
    with torch.no_grad():
        for xb, yb, mb in val_loader:
            xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
            corrected = apply_correction(xb, model(xb))
            se += (((corrected - yb)**2) * mb).sum().item()
            base_se += (((xb - yb)**2) * mb).sum().item()
            n += mb.sum().item()
    rmse_model, rmse_base = (se/n)**0.5, (base_se/n)**0.5
    skill = (1 - rmse_model/rmse_base) * 100
    flag = ''
    if rmse_model < best:
        best = rmse_model; torch.save(model.state_dict(), 'isih_prototype.pt'); flag = ' *saved*'
    print(f'epoch {epoch:2d}  model RMSE {rmse_model:.4f}  background {rmse_base:.4f}  '
          f'improvement {skill:+.1f}%{flag}')

## 8. Result

The number that matters: does the model beat the background it corrects? If improvement is not positive, the honest outcome is to report that and ship the raw background — which is exactly the fallback the architecture already specifies.

In [ ]:
# weights_only=True: this file holds tensors only, and the flag keeps
# torch from unpickling arbitrary objects.
model.load_state_dict(torch.load('isih_prototype.pt', weights_only=True))
model.eval()
se = base_se = n = 0.0
with torch.no_grad():
    for xb, yb, mb in val_loader:
        xb, yb, mb = xb.to(device), yb.to(device), mb.to(device)
        corrected = apply_correction(xb, model(xb))
        se += (((corrected - yb)**2) * mb).sum().item()
        base_se += (((xb - yb)**2) * mb).sum().item()
        n += mb.sum().item()
rmse_model, rmse_base = (se/n)**0.5, (base_se/n)**0.5
print(f'held-out background RMSE : {rmse_base:.4f}')
print(f'held-out model RMSE      : {rmse_model:.4f}')
print(f'improvement              : {(1-rmse_model/rmse_base)*100:+.1f}%')
print()
if rmse_model < rmse_base:
    print('Model beats its background on held-out data. Download isih_prototype.pt')
else:
    print('Model does NOT beat the background. Report this honestly and use the')
    print('raw background for the demo -- that is the specified fallback, not a failure.')